# Kintsugi Experiment Results Summary

This notebook consolidates results from all experiments conducted in this project, providing purpose, results (CSV data), and conclusions for each phase.

## Table of Contents
1. [Exp 01] Prompt Strategy Comparison
2. [Exp 02] Mental Model Proof of Concept
3. [Exp 06] Batch Student Analysis (30 Students)
4. [Exp 07] Validation Analysis (Grade Correlation & Baseline)
5. [Exp 08] Consistency Test
6. [Exp 09] PFA Knowledge Tracing Comparison
7. [Exp 10] Q-Matrix Refinement Analysis
8. [Exp 11] Human-LLM Agreement Analysis


### 🔵 [Exp No. 01] Prompt Strategy Comparison

**Purpose Of Exp:**
Compare different LLM prompting strategies (Zero-Shot, CoT, Few-Shot) to determine the most effective method for identifying student Knowledge Components (KCs) from their code submissions.

**Conclusion:**
Context-enriched prompts that include both the problem description and previous student history (mental model) significantly improved accuracy over zero-shot baselines.


In [1]:
import pandas as pd
import os
import json

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

def find_results_path(path_within_results):
    for prefix in ['', '../', '../../', '../../../']:
        full_path = os.path.join(prefix, 'results', path_within_results)
        if os.path.exists(full_path):
            return full_path
    
    # Also check dataset folder if not found in results
    for prefix in ['', '../', '../../', '../../../']:
        full_path = os.path.join(prefix, 'dataset', path_within_results)
        if os.path.exists(full_path):
            return full_path
    return None

def display_file(relative_path):
    full_path = find_results_path(relative_path)
    if not full_path:
        return

    print("="*80)
    print(f"FILE: {relative_path}")
    print("-"*80)

    if relative_path.endswith('.csv'):
        df = pd.read_csv(full_path)
        # Standard pandas display for notebooks (clean table)
        display(df.head(10))
        
    elif relative_path.endswith('.json'):
        with open(full_path, 'r') as f:
            data = json.load(f)
            
        if isinstance(data, dict):
            # Print simple key-value pairs like a terminal
            for k, v in data.items():
                if not isinstance(v, (dict, list)):
                    print(f"{k:30}: {v}")
            
            # Show nested structures as clean tables if possible
            for k, v in data.items():
                if isinstance(v, (dict, list)):
                    print(f"\n[{k}]")
                    try:
                        display(pd.DataFrame(v).head(5))
                    except:
                        print(json.dumps(v, indent=4))
        elif isinstance(data, list):
            try:
                display(pd.DataFrame(data).head(10))
            except:
                print(json.dumps(data, indent=4))
    print("="*80 + "\n")

# Results for Exp 01: Prompt Strategy Comparison
display_file('01_prompt_strategy_comparison/single_student_14355_exp4_summary.csv')
display_file('01_prompt_strategy_comparison/single_student_14355_exp3_results.csv')


FILE: 01_prompt_strategy_comparison/single_student_14355_exp4_summary.csv
--------------------------------------------------------------------------------


,Strategy,Valid_Responses,Total_Responses,Coverage_Pct,Avg_Time_Sec
0,Few-Shot,5,5,100.0,8.442
1,Zero-Shot,5,5,100.0,10.719
2,Chain-of-Thought,5,5,100.0,12.827
3,Curriculum-Aware,5,5,100.0,14.703



FILE: 01_prompt_strategy_comparison/single_student_14355_exp3_results.csv
--------------------------------------------------------------------------------


,SubjectID,ProblemID,Score,Code,Zero-Shot_Output,Zero-Shot_TimeSec,Few-Shot_Output,Few-Shot_TimeSec,Chain-of-Thought_Output,Chain-of-Thought_TimeSec,Curriculum-Aware_Output,Curriculum-Aware_TimeSec
0,14355,13,0.782609,"public int caughtSpeeding(int speed, boolean i...",{'knowledge_gaps': ['Conditional Logic Optimiz...,16.424,"{'knowledge_gaps': [""Misinterpretation of prob...",12.932,{'reasoning_chain': {'step1_behavior': 'The co...,14.725,"{'student_analysis': [{'student_id': '14355', ...",27.840
1,14355,24,0.000000,"public int blackjack(int a, int b)\r\n{\r\n ...",{'knowledge_gaps': ['Method Return Types: The ...,11.098,{'knowledge_gaps': ['Method Signature and Retu...,7.890,{'reasoning_chain': {'step1_behavior': 'The `b...,18.780,"{'student_analysis': [{'student_id': '14355', ...",10.488
2,14355,40,0.076923,public String getSandwich(String str)\r\n{\r\n...,"{'knowledge_gaps': [""Understanding problem req...",8.498,{'knowledge_gaps': ['Problem decomposition and...,6.531,"{'reasoning_chain': {'step1_behavior': ""The st...",9.599,"{'student_analysis': [{'student_id': '14355', ...",8.523
3,14355,41,0.000000,public int sum3(int[] nums)\r\n{\r\n return ...,{'knowledge_gaps': ['Basic Java syntax for arr...,6.891,{'knowledge_gaps': ['Fundamental Java syntax f...,6.828,{'reasoning_chain': {'step1_behavior': 'The `s...,8.191,"{'student_analysis': [{'student_id': '14355', ...",11.133
4,14355,232,0.000000,"public String alarmClock(int day, boolean vaca...",{'knowledge_gaps': ['**Method Return Types and...,10.682,{'knowledge_gaps': ['Recursion: Missing base c...,8.028,{'reasoning_chain': {'step1_behavior': 'The `a...,12.841,"{'student_analysis': [{'student_id': '14355', ...",15.532


### 🟢 [Exp No. 05] Mental Model Comparison

**Purpose Of Exp:**
Evaluate the quality of the generated student mental models by comparing different students (student 10155, 14359, 14475, 14374) using simple average metrics and context-enriched prompts.

**Conclusion:**
Students' mental models accurately reflect their unique learning trajectories, with context-enriched prompts providing a more nuanced understanding of their individual strengths and weaknesses compared to simple averages.


In [2]:
# Results for Exp 05: Mental Model Comparison
exp05_inner_path = '05_mental_model_comparison/v1_simple_average/'
exp05_path = find_results_path(exp05_inner_path)

if exp05_path:
    for csv_file in sorted(os.listdir(exp05_path)):
        if csv_file.endswith('.csv'):
            display_file(os.path.join(exp05_inner_path, csv_file))
else:
    print(f"Path not found: {exp05_inner_path}")


FILE: 05_mental_model_comparison/v1_simple_average/mental_model_comparison_student_10155.csv
--------------------------------------------------------------------------------


,SubjectID,ProblemID,Score,ScorePct,IsPerfect,Expected_KCTags,WeakSkills,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_KCTags,Enriched_KCTags,Baseline_TimeSec,Enriched_TimeSec,Baseline_PerfectCorrect,Enriched_PerfectCorrect,Baseline_Relevance_F1,Enriched_Relevance_F1
0,10155,1,1.000000,100.0000,True,"['If/Else', 'Math+-*/', 'LogicAndNotOr', 'Logi...","['While', 'DefFunction', 'StringConcat', 'Math...",0,0,0.0000,0.0000,[],[],4.367,2.983,True,True,0.0000,0.0000
1,10155,20,1.000000,100.0000,True,"['If/Else', 'NestedIf', 'Math+-*/', 'LogicAndN...","['While', 'DefFunction', 'StringConcat', 'Math...",0,0,0.0000,0.0000,[],[],5.497,11.121,True,True,0.0000,0.0000
2,10155,232,1.000000,100.0000,True,"['If/Else', 'NestedIf', 'LogicAndNotOr', 'Logi...","['While', 'DefFunction', 'StringConcat', 'Math...",2,2,0.1176,0.2222,"['If/Else', 'LogicAndNotOr', 'NestedIf']","['DefFunction', 'If/Else', 'LogicAndNotOr', 'N...",7.515,19.961,False,False,0.6667,0.6000
3,10155,233,1.000000,100.0000,True,"['If/Else', 'Math+-*/', 'LogicAndNotOr', 'Logi...","['While', 'DefFunction', 'StringConcat', 'Math...",0,0,0.0000,0.0000,[],[],4.599,6.461,True,True,0.0000,0.0000
4,10155,235,0.857143,85.7143,False,"['If/Else', 'LogicAndNotOr', 'LogicCompareNum']","['While', 'DefFunction', 'StringConcat', 'Math...",3,4,0.1111,0.1176,"['If/Else', 'LogicAndNotOr', 'LogicBoolean', '...","['If/Else', 'LogicAndNotOr', 'NestedIf']",21.160,20.157,NaN,NaN,0.5714,0.6667
5,10155,236,0.750000,75.0000,False,"['If/Else', 'LogicAndNotOr', 'LogicCompareNum']","['While', 'DefFunction', 'StringConcat', 'Math...",2,2,0.1176,0.1176,"['If/Else', 'LogicAndNotOr', 'NestedIf']","['If/Else', 'LogicAndNotOr', 'NestedIf']",16.292,22.873,NaN,NaN,0.6667,0.6667
6,10155,106,0.625000,62.5000,False,"['If/Else', 'For', 'NestedFor', 'LogicAndNotOr...","['While', 'DefFunction', 'StringConcat', 'Math...",4,4,0.4211,0.6000,"['ArrayIndex', 'For', 'If/Else', 'LogicBoolean...","['ArrayIndex', 'DefFunction', 'For', 'If/Else'...",14.390,12.155,NaN,NaN,0.7273,0.8333
7,10155,46,0.555556,55.5556,False,"['If/Else', 'For', 'LogicAndNotOr', 'LogicComp...","['While', 'DefFunction', 'StringConcat', 'Math...",4,3,0.5000,0.6000,"['ArrayIndex', 'DefFunction', 'For', 'If/Else'...","['ArrayIndex', 'DefFunction', 'For', 'NestedFo...",11.859,18.820,NaN,NaN,0.7273,0.3636
8,10155,36,0.411765,41.1765,False,"['If/Else', 'For', 'Math+-*/', 'LogicAndNotOr'...","['While', 'DefFunction', 'StringConcat', 'Math...",4,6,0.5714,0.6364,"['CharEqual', 'DefFunction', 'For', 'If/Else',...","['ArrayIndex', 'CharEqual', 'DefFunction', 'Fo...",7.495,14.091,NaN,NaN,0.6667,0.5000
9,10155,49,0.411765,41.1765,False,"['If/Else', 'For', 'Math+-*/', 'LogicCompareNu...","['While', 'DefFunction', 'StringConcat', 'Math...",5,3,0.5263,0.6000,"['ArrayIndex', 'DefFunction', 'For', 'If/Else'...","['ArrayIndex', 'DefFunction', 'For', 'If/Else'...",15.181,10.461,NaN,NaN,0.8000,0.5455



FILE: 05_mental_model_comparison/v1_simple_average/mental_model_comparison_student_14359.csv
--------------------------------------------------------------------------------


,SubjectID,ProblemID,Score,ScorePct,IsPerfect,Expected_KCTags,WeakSkills,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_KCTags,Enriched_KCTags,Baseline_TimeSec,Enriched_TimeSec,Baseline_PerfectCorrect,Enriched_PerfectCorrect,Baseline_Relevance_F1,Enriched_Relevance_F1
0,14359,13,1.000000,100.0000,True,"['If/Else', 'NestedIf', 'Math+-*/', 'LogicAndN...","['StringConcat', 'While']",0,0,0.0,0.0000,[],[],4.459,3.756,True,True,0.0000,0.0000
1,14359,24,1.000000,100.0000,True,"['If/Else', 'Math+-*/', 'LogicAndNotOr', 'Logi...","['StringConcat', 'While']",3,2,0.0,0.0000,"['If/Else', 'LogicCompareNum', 'NestedIf']","['If/Else', 'LogicAndNotOr', 'LogicCompareNum'...",34.985,36.813,False,False,0.5714,0.7500
2,14359,37,1.000000,100.0000,True,"['If/Else', 'LogicCompareNum', 'StringFormat',...","['StringConcat', 'While']",0,0,0.0,0.0000,[],[],3.729,4.562,True,True,0.0000,0.0000
3,14359,56,1.000000,100.0000,True,"['If/Else', 'For', 'Math+-*/', 'LogicAndNotOr'...","['StringConcat', 'While']",0,0,0.0,0.0000,[],[],5.493,5.474,True,True,0.0000,0.0000
4,14359,57,1.000000,100.0000,True,"['For', 'ArrayIndex']","['StringConcat', 'While']",0,0,0.0,0.0000,[],[],1.995,3.184,True,True,0.0000,0.0000
5,14359,232,1.000000,100.0000,True,"['If/Else', 'NestedIf', 'LogicAndNotOr', 'Logi...","['StringConcat', 'While']",0,0,0.0,0.0000,[],[],4.479,4.456,True,True,0.0000,0.0000
6,14359,235,1.000000,100.0000,True,"['If/Else', 'LogicAndNotOr', 'LogicCompareNum']","['StringConcat', 'While']",0,0,0.0,0.0000,[],[],7.031,5.737,True,True,0.0000,0.0000
7,14359,108,0.684211,68.4211,False,"['If/Else', 'For', 'LogicAndNotOr', 'LogicComp...","['StringConcat', 'While']",2,3,0.0,0.0000,"['ArrayIndex', 'For', 'LogicBoolean', 'NestedF...","['ArrayIndex', 'For', 'If/Else', 'LogicAndNotO...",37.161,41.798,NaN,NaN,0.4444,0.8000
8,14359,32,0.545455,54.5455,False,"['If/Else', 'While', 'LogicCompareNum', 'Strin...","['StringConcat', 'While']",2,3,0.0,0.5714,"['For', 'If/Else', 'StringIndex']","['For', 'If/Else', 'StringConcat', 'StringInde...",13.314,24.114,NaN,NaN,0.3636,0.6154
9,14359,107,0.545455,54.5455,False,"['If/Else', 'For', 'Math+-*/', 'LogicAndNotOr'...","['StringConcat', 'While']",2,1,0.0,0.4000,"['For', 'LogicAndNotOr', 'LogicBoolean', 'Nest...","['For', 'NestedFor', 'While']",11.679,25.441,NaN,NaN,0.4000,0.2222



FILE: 05_mental_model_comparison/v1_simple_average/mental_model_comparison_student_14475.csv
--------------------------------------------------------------------------------


,SubjectID,ProblemID,Score,ScorePct,IsPerfect,Expected_KCTags,WeakSkills,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_KCTags,Enriched_KCTags,Baseline_TimeSec,Enriched_TimeSec,Baseline_PerfectCorrect,Enriched_PerfectCorrect,Baseline_Relevance_F1,Enriched_Relevance_F1
0,14475,22,1.000000,100.0000,True,"['If/Else', 'NestedIf', 'Math+-*/', 'LogicAndN...",[],0,0,0.0,0.0,[],[],1.803,4.556,True,True,0.0,0.0000
1,14475,32,1.000000,100.0000,True,"['If/Else', 'While', 'LogicCompareNum', 'Strin...",[],0,0,0.0,0.0,[],[],3.006,6.436,True,True,0.0,0.0000
2,14475,49,1.000000,100.0000,True,"['If/Else', 'For', 'Math+-*/', 'LogicCompareNu...",[],0,0,0.0,0.0,[],[],7.262,4.323,True,True,0.0,0.0000
3,14475,51,1.000000,100.0000,True,"['If/Else', 'For', 'Math%', 'LogicCompareNum',...",[],0,0,0.0,0.0,[],[],2.759,6.402,True,True,0.0,0.0000
4,14475,56,1.000000,100.0000,True,"['If/Else', 'For', 'Math+-*/', 'LogicAndNotOr'...",[],0,0,0.0,0.0,[],[],5.738,21.189,True,True,0.0,0.0000
5,14475,57,1.000000,100.0000,True,"['For', 'ArrayIndex']",[],0,0,0.0,0.0,[],[],3.164,2.924,True,True,0.0,0.0000
6,14475,128,1.000000,100.0000,True,"['If/Else', 'For', 'LogicAndNotOr', 'StringFor...",[],0,0,0.0,0.0,[],[],3.508,3.874,True,True,0.0,0.0000
7,14475,235,1.000000,100.0000,True,"['If/Else', 'LogicAndNotOr', 'LogicCompareNum']",[],0,0,0.0,0.0,[],[],5.175,4.500,True,True,0.0,0.0000
8,14475,38,0.933333,93.3333,False,"['If/Else', 'For', 'LogicAndNotOr', 'StringFor...",[],1,1,0.0,0.0,"['For', 'LogicAndNotOr', 'StringIndex']","['For', 'StringIndex']",7.938,17.089,NaN,NaN,0.6,0.4444
9,14475,118,0.909091,90.9091,False,"['For', 'ArrayIndex']",[],1,1,0.0,0.0,"['ArrayIndex', 'If/Else']","['ArrayIndex', 'If/Else']",8.499,11.594,NaN,NaN,0.5,0.5000



FILE: 05_mental_model_comparison/v1_simple_average/mental_model_context_enriched_student_14374.csv
--------------------------------------------------------------------------------


,SubjectID,ProblemID,Score,ContextEnriched_Output,ContextEnriched_TimeSec,ContextEnriched_WeakSkillOverlap
0,14374,232,0.000000,"{'student_analysis': [{'student_id': '14374', ...",10.032,0
1,14374,41,1.000000,"{'student_analysis': [{'student_id': '14374', ...",4.791,0
2,14374,67,0.550000,"{'student_analysis': [{'student_id': '14374', ...",11.635,0
3,14374,44,1.000000,"{'student_analysis': [{'student_id': '14374', ...",10.519,0
4,14374,25,0.761905,"{'student_analysis': [{'student_id': '14374', ...",8.478,0
5,14374,28,0.000000,"{'student_analysis': [{'student_id': '14374', ...",13.770,0


### 🟠 [Exp No. 06] Batch Student Analysis (30 Students)

**Purpose Of Exp:**
Scale the mental model generation and KC identification across a larger cohort of 30 students to evaluate consistency and scalability.

**Conclusion:**
Performance remained stable across the larger cohort, with minor variations based on student performance levels (struggling vs high performers). Corrected F1 metrics showed improved reliability.


In [3]:
# Results for Exp 06: Batch Student Analysis
display_file('06_batch_30students/batch_comparison_30students_corrected.csv')
display_file('06_batch_30students/per_student_summary_corrected.csv')
display_file('06_batch_30students/metadata.json')

# Checkpoints
checkpoint_inner = '06_batch_30students/'
checkpoint_path = find_results_path(checkpoint_inner)
if checkpoint_path:
    for cp in sorted(os.listdir(checkpoint_path)):
        if cp.startswith('checkpoint_') and cp.endswith('.csv'):
            display_file(f'06_batch_30students/{cp}')


FILE: 06_batch_30students/batch_comparison_30students_corrected.csv
--------------------------------------------------------------------------------


,SubjectID,Cluster,NumWeakSkills,ProblemID,Score,ScorePct,IsPerfect,Baseline_KCTags,Enriched_KCTags,Baseline_GAP_Count,...,Corrected_Baseline_Recall,Corrected_Enriched_Precision,Corrected_Enriched_Recall,Num_Testable_Weak,Testable_Weak_Skills,Problem_Required_Skills,Baseline_Relevance_F1,Enriched_Relevance_F1,Baseline_TimeSec,Enriched_TimeSec
0,106,Average,0,1,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicAndNotOr', 'LogicCompareNum'...",0.0,0.0,2.850,3.776
1,106,Average,0,17,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicAndNotOr', 'LogicCompareNum'...",0.0,0.0,9.396,9.683
2,106,Average,0,100,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['DefFunction', 'If/Else', 'LogicCompareNum', ...",0.0,0.0,3.832,4.733
3,106,Average,0,3,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicAndNotOr', 'LogicBoolean', '...",0.0,0.0,3.832,4.409
4,106,Average,0,102,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicCompareNum', 'StringEqual', ...",0.0,0.0,3.811,5.237
5,106,Average,0,233,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicAndNotOr', 'LogicCompareNum'...",0.0,0.0,3.488,4.248
6,106,Average,0,5,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicBoolean', 'NestedIf']",0.0,0.0,3.679,3.757
7,106,Average,0,235,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicAndNotOr', 'LogicCompareNum']",0.0,0.0,6.114,5.734
8,106,Average,0,101,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['If/Else', 'LogicAndNotOr', 'LogicCompareNum'...",0.0,0.0,5.682,7.857
9,106,Average,0,22,1.0,100.0,True,[],[],0,...,0.0,0.0,0.0,0,[],"['DefFunction', 'If/Else', 'LogicAndNotOr', 'L...",0.0,0.0,4.554,5.910



FILE: 06_batch_30students/per_student_summary_corrected.csv
--------------------------------------------------------------------------------


,SubjectID,Cluster,NumWeakSkills,TotalProblems,TestableProblems,Old_Baseline_F1,Old_Enriched_F1,Old_Delta,Corrected_Baseline_F1,Corrected_Enriched_F1,Corrected_Delta
0,14359,Average,2,12,2,0.0516,0.1032,0.0516,0.1666,0.4857,0.3191
1,10083,Average,1,12,1,0.0278,0.1266,0.0988,0.3333,0.4000,0.0667
2,14316,Average,1,12,1,0.0611,0.0944,0.0333,0.3333,0.4000,0.0667
3,14476,Average,3,12,2,0.0417,0.1124,0.0708,0.2679,0.3095,0.0416
4,106,Average,0,12,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,10224,Average,1,12,2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
6,14381,Average,0,10,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
7,14429,Average,0,10,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
8,14459,Average,0,12,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
9,14186,Average,1,12,1,0.1083,0.0840,-0.0243,0.4000,0.2222,-0.1778



FILE: 06_batch_30students/metadata.json
--------------------------------------------------------------------------------
experiment                    : 06_batch_mental_model_comparison
run_timestamp                 : 2026-03-19T18:45:02.640842
model_id                      : gemini-2.5-flash
total_students                : 28
total_problems                : 330
students_per_cluster          : 10
overall_baseline_f1           : 0.1535
overall_enriched_f1           : 0.1987
overall_delta_f1              : 0.0452
students_improved             : 11
students_same                 : 16
students_worse                : 1

FILE: 06_batch_30students/checkpoint_10_students.csv
--------------------------------------------------------------------------------


...

,SubjectID,Cluster,NumWeakSkills,ProblemID,Score,ScorePct,IsPerfect,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_Relevance_F1,Enriched_Relevance_F1,Baseline_KCTags,Enriched_KCTags,Baseline_TimeSec,Enriched_TimeSec,Baseline_PerfectCorrect,Enriched_PerfectCorrect
0,106,Average,0,1,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],2.850,3.776,True,True
1,106,Average,0,17,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],9.396,9.683,True,True
2,106,Average,0,100,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.832,4.733,True,True
3,106,Average,0,3,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.832,4.409,True,True
4,106,Average,0,102,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.811,5.237,True,True
5,106,Average,0,233,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.488,4.248,True,True
6,106,Average,0,5,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.679,3.757,True,True
7,106,Average,0,235,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],6.114,5.734,True,True
8,106,Average,0,101,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],5.682,7.857,True,True
9,106,Average,0,22,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],4.554,5.910,True,True



FILE: 06_batch_30students/checkpoint_25_students.csv
--------------------------------------------------------------------------------


,SubjectID,Cluster,NumWeakSkills,ProblemID,Score,ScorePct,IsPerfect,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_Relevance_F1,Enriched_Relevance_F1,Baseline_KCTags,Enriched_KCTags,Baseline_TimeSec,Enriched_TimeSec,Baseline_PerfectCorrect,Enriched_PerfectCorrect
0,106,Average,0,1,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],2.850,3.776,True,True
1,106,Average,0,17,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],9.396,9.683,True,True
2,106,Average,0,100,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.832,4.733,True,True
3,106,Average,0,3,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.832,4.409,True,True
4,106,Average,0,102,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.811,5.237,True,True
5,106,Average,0,233,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.488,4.248,True,True
6,106,Average,0,5,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.679,3.757,True,True
7,106,Average,0,235,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],6.114,5.734,True,True
8,106,Average,0,101,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],5.682,7.857,True,True
9,106,Average,0,22,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],4.554,5.910,True,True



FILE: 06_batch_30students/checkpoint_5_students.csv
--------------------------------------------------------------------------------


,SubjectID,Cluster,NumWeakSkills,ProblemID,Score,ScorePct,IsPerfect,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1,Baseline_Relevance_F1,Enriched_Relevance_F1,Baseline_KCTags,Enriched_KCTags,Baseline_TimeSec,Enriched_TimeSec,Baseline_PerfectCorrect,Enriched_PerfectCorrect
0,106,Average,0,1,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],2.850,3.776,True,True
1,106,Average,0,17,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],9.396,9.683,True,True
2,106,Average,0,100,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.832,4.733,True,True
3,106,Average,0,3,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.832,4.409,True,True
4,106,Average,0,102,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.811,5.237,True,True
5,106,Average,0,233,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.488,4.248,True,True
6,106,Average,0,5,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],3.679,3.757,True,True
7,106,Average,0,235,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],6.114,5.734,True,True
8,106,Average,0,101,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],5.682,7.857,True,True
9,106,Average,0,22,1.0,100.0,True,0,0,0.0,0.0,0.0,0.0,[],[],4.554,5.910,True,True


### 🟣 [Exp No. 07] Validation Analysis (Grade Correlation & Baseline)

**Purpose Of Exp:**
1.  **Grade Correlation**: Determine if students with more identified weak skills (KCs) received lower final grades.
2.  **Random Baseline**: Compare LLM results against a random chance baseline to validate performance significance.
3.  **Relevance Analysis**: Ensure the identified KCs are relevant to the problem descriptions.

**Conclusion:**
There was a statistically significant negative correlation between weak KCs and final grades. Results also outperformed the random baseline significantly across all student profiles.


In [4]:
# Results for Exp 07: Validation Analysis
display_file('07_validation_analysis/grade_correlation_data.csv')
display_file('07_validation_analysis/metadata.json')


FILE: 07_validation_analysis/grade_correlation_data.csv
--------------------------------------------------------------------------------


,SubjectID,AvgScore,NumProblems,NumWeakSkills,XGrade
0,106,0.9905,20,0,0.8100
1,263,1.0000,40,0,0.3600
2,714,0.9649,29,0,0.4522
3,2253,1.0000,50,0,0.4000
4,2411,1.0000,40,0,0.5200
5,2621,1.0000,50,0,0.4200
6,4419,1.0000,41,0,0.5951
7,4564,0.9487,39,0,0.9200
8,4840,0.8355,16,2,0.8100
9,6493,1.0000,50,0,0.8500



FILE: 07_validation_analysis/metadata.json
--------------------------------------------------------------------------------
experiment                    : 07_validation_analysis
run_timestamp                 : 2026-03-18T17:54:13.208500

[sections]


,7A_grade_correlation,7B_relevance_analysis,7C_random_baseline
students_analyzed,372,NaN,NaN
pearson_r_weakskills_vs_grade,-0.0749,NaN,NaN
p_value,0.149466,NaN,NaN
significant,False,NaN,NaN
students,NaN,[],NaN


### 🔴 [Exp No. 08] Consistency Test

**Purpose Of Exp:**
Test the LLM’s consistency by repeatedly running assessment on the same students over time to check how stable it identifies student knowledge levels.

**Conclusion:**
The model shows high consistency with low standard deviation across multiple runs, indicating reliable performance for longitudinal student tracking.


In [5]:
# Results for Exp 08: Consistency Test
display_file('08_consistency_test/consistency_results.csv')
display_file('08_consistency_test/metadata.json')


FILE: 08_consistency_test/consistency_results.csv
--------------------------------------------------------------------------------


,ProblemID,ScorePct,Condition,Run1_Tags,Run2_Tags,Run3_Tags,Core_Tags,Variable_Tags,Num_Core,Num_Variable,Avg_Jaccard
0,108,68.4211,Baseline,"['ArrayIndex', 'If/Else', 'NestedFor']","['ArrayIndex', 'LogicBoolean', 'NestedFor']","['ArrayIndex', 'LogicBoolean', 'NestedFor']","['ArrayIndex', 'NestedFor']","['If/Else', 'LogicBoolean']",2,2,0.6667
1,108,68.4211,Enriched,"['ArrayIndex', 'For', 'If/Else', 'NestedFor', ...","['ArrayIndex', 'For', 'NestedFor', 'While']","['ArrayIndex', 'If/Else', 'LogicAndNotOr', 'Ne...","['ArrayIndex', 'NestedFor']","['For', 'If/Else', 'LogicAndNotOr', 'While']",2,4,0.5444
2,32,54.5455,Baseline,"['For', 'If/Else', 'StringEqual', 'StringIndex']","['For', 'If/Else', 'StringIndex']","['For', 'If/Else', 'StringIndex', 'While']","['For', 'If/Else', 'StringIndex']","['StringEqual', 'While']",3,2,0.7000
3,32,54.5455,Enriched,"['For', 'If/Else', 'StringConcat', 'StringInde...","['For', 'If/Else', 'StringIndex', 'While']","['For', 'If/Else', 'StringConcat', 'StringIndex']","['For', 'If/Else', 'StringIndex']","['StringConcat', 'While']",3,2,0.7333
4,107,54.5455,Baseline,"['For', 'LogicBoolean', 'NestedFor']","['ArrayIndex', 'For', 'LogicBoolean', 'NestedF...","['For', 'LogicBoolean', 'While']","['For', 'LogicBoolean']","['ArrayIndex', 'NestedFor', 'While']",2,3,0.5500
5,107,54.5455,Enriched,"['ArrayIndex', 'NestedFor', 'While']","['For', 'NestedFor', 'While']","['For', 'LogicBoolean', 'NestedFor', 'While']","['NestedFor', 'While']","['ArrayIndex', 'For', 'LogicBoolean']",2,3,0.5500
6,40,38.4615,Baseline,"['DefFunction', 'For', 'LogicAndNotOr', 'Strin...","['For', 'If/Else', 'StringEqual', 'StringIndex']","['DefFunction', 'For', 'LogicAndNotOr', 'Strin...","['For', 'StringEqual', 'StringIndex']","['DefFunction', 'If/Else', 'LogicAndNotOr']",3,3,0.6667
7,40,38.4615,Enriched,"['For', 'If/Else', 'NestedFor', 'StringEqual',...","['ArrayIndex', 'DefFunction', 'For', 'LogicAnd...","['DefFunction', 'For', 'LogicAndNotOr', 'Strin...","['For', 'StringEqual', 'StringIndex']","['ArrayIndex', 'DefFunction', 'If/Else', 'Logi...",3,5,0.5456
8,34,14.2857,Baseline,"['For', 'NestedFor', 'StringConcat', 'StringIn...","['For', 'If/Else', 'LogicBoolean', 'StringConc...","['For', 'If/Else', 'LogicBoolean', 'StringConc...","['For', 'StringConcat', 'StringIndex']","['If/Else', 'LogicBoolean', 'NestedFor']",3,3,0.6667
9,34,14.2857,Enriched,"['For', 'LogicBoolean', 'NestedFor', 'StringCo...","['ArrayIndex', 'For', 'If/Else', 'NestedFor', ...","['For', 'NestedFor', 'StringConcat', 'StringIn...","['For', 'NestedFor', 'StringConcat']","['ArrayIndex', 'If/Else', 'LogicBoolean', 'Str...",3,5,0.5139



FILE: 08_consistency_test/metadata.json
--------------------------------------------------------------------------------
experiment                    : 08_consistency_test
run_timestamp                 : 2026-03-17T12:27:26.448208
model_id                      : gemini-2.5-flash
student_id                    : 14359
num_runs                      : 3
overall_jaccard               : 0.6137
verdict                       : MODERATE
baseline_avg_jaccard          : 0.65
enriched_avg_jaccard          : 0.5774

[problems_tested]


,0
0,108
1,32
2,107
3,40
4,34



[conditions]


,0
0,Baseline
1,Enriched


### 🟤 [Exp No. 09] PFA Knowledge Tracing Comparison

**Purpose Of Exp:**
Compare our LLM-enhanced KC mappings with Instructor KCs and regular LLM-generated KCs using Performance Factor Analysis (PFA). Evaluate prediction accuracy (AUC) of student performance.

**Conclusion:**
Enhanced KCs (LLM refined + instructor) provided the highest AUC, indicating improved tracking of student learning over time.


In [6]:
# Results for Exp 09: PFA Knowledge Tracing Comparison
display_file('09_pfa_comparison/09_pfa_results.csv')
display_file('09_pfa_comparison/12_pfa_additive_results.csv')


FILE: 09_pfa_comparison/09_pfa_results.csv
--------------------------------------------------------------------------------


,KC_Model,Num_KCs,AUC_mean,AUC_std,F1_mean,F1_std,Acc_mean,Acc_std
0,KCGen-KT LLM (11 KCs),11,0.757322,0.007507,0.552362,0.008367,0.709243,0.005651
1,Instructor (18 KCs),18,0.773748,0.006517,0.587887,0.003658,0.721464,0.007916
2,Enhanced (17 KCs),18,0.765840,0.007921,0.587458,0.012414,0.720608,0.008693



FILE: 09_pfa_comparison/12_pfa_additive_results.csv
--------------------------------------------------------------------------------


,KC_Model,Num_KCs,AUC_mean,AUC_std,F1_mean,F1_std,Acc_mean,Acc_std
0,KCGen-KT LLM (11 KCs),11,0.757322,0.007507,0.552362,0.008367,0.709243,0.005651
1,Instructor (18 KCs),18,0.773748,0.006517,0.587887,0.003658,0.721464,0.007916
2,Additive/10b (18 KCs),18,0.770343,0.012030,0.579531,0.010258,0.720399,0.010898


### 💠 [Exp No. 10] Q-Matrix Refinement Analysis

**Purpose Of Exp:**
Experiment with refining the problem-to-KC mapping (Q-Matrix) using LLM suggestions and analyze its effect on PFA prediction performance.

**Conclusion:**
Refining the Q-matrix by either adding missing KCs or removing irrelevant ones improves accuracy, suggesting current instructor-curated KCs may have gaps that LLMs can help fill.


In [7]:
# Results for Exp 10: Q-Matrix Refinement
display_file('10_qmatrix_refinement/qmatrix_refinement_results.csv')
display_file('10_qmatrix_refinement/qmatrix_additive_results.csv')


FILE: 10_qmatrix_refinement/qmatrix_refinement_results.csv
--------------------------------------------------------------------------------


,ProblemID,Instructor_KCs,Refined_KCs,Removed_KCs,Added_KCs,Reasoning,Num_Instructor,Num_Refined,Num_Removed,Num_Added,Raw_Response
0,1,"[""If/Else"", ""Math+-*/"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""Math+-*/"", ""LogicAndNotOr"", ""Logi...",[],[],The failing students either completely omit th...,4,4,0,0,"```json\n{\n ""problem_id"": 1,\n ""refined..."
1,3,"[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""...","[""NestedIf""]",[],The problem primarily tests the ability to use...,5,4,1,0,"```json\n{\n ""problem_id"": 3,\n ""refined..."
2,5,"[""If/Else"", ""NestedIf"", ""LogicBoolean""]","[""If/Else"", ""LogicBoolean"", ""LogicAndNotOr""]","[""NestedIf""]",[],The core issue differentiating passing and fai...,3,3,1,0,"```json\n{\n ""problem_id"": 5,\n ""refined..."
3,12,"[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""...","[""NestedIf""]",[],The problem primarily tests the ability to use...,5,4,1,0,"```json\n{\n ""problem_id"": 12,\n ""refine..."
4,13,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""...","[""NestedIf""]",[],The core issue differentiating passing and fai...,6,5,1,0,"```json\n{\n ""problem_id"": 13,\n ""refine..."
5,17,"[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""]","[""NestedIf""]",[],The problem primarily tests the ability to con...,4,3,1,0,"```json\n{\n ""problem_id"": 17,\n ""refine..."
6,20,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""...","[""NestedIf""]",[],The core issue is correctly handling the condi...,5,5,1,0,"```json\n{\n ""problem_id"": 20,\n ""refine..."
7,21,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""If/Else"", ""Math+-*/"", ""LogicCompareNum""]","[""NestedIf"", ""LogicAndNotOr""]",[],The core issue is correctly implementing the c...,5,3,2,0,"```json\n{\n ""problem_id"": 21,\n ""refine..."
8,22,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""...","[""NestedIf"", ""Math+-*/""]",[],The core issue differentiating passing and fai...,6,4,2,0,"```json\n{\n ""problem_id"": 22,\n ""refine..."
9,24,"[""If/Else"", ""Math+-*/"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""LogicAndNotOr"", ""LogicCompareNum""...",[],[],The core logic involves multiple if/else state...,4,4,0,0,"```json\n{\n ""problem_id"": 24,\n ""refine..."



FILE: 10_qmatrix_refinement/qmatrix_additive_results.csv
--------------------------------------------------------------------------------


,ProblemID,Instructor_KCs,Final_KCs,Added_KCs,Reasoning,Num_Instructor,Num_Final,Num_Added,TimeSec,Raw_Response
0,1,"[""If/Else"", ""Math+-*/"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""Math+-*/"", ""LogicAndNotOr"", ""Logi...","[""If/Else""]",Failing Student 1 completely omits the if/else...,4,4,1,1.85,"```json\n{\n ""problem_id"": 1,\n ""added_k..."
1,3,"[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...","[""NestedIf"", ""LogicAndNotOr"", ""LogicCompareNum...",[],The instructor's tags are sufficient. The fail...,5,5,0,1.90,"```json\n{\n ""problem_id"": 3,\n ""added_k..."
2,5,"[""If/Else"", ""NestedIf"", ""LogicBoolean""]","[""NestedIf"", ""LogicAndNotOr"", ""LogicCompareNum...","[""LogicAndNotOr"", ""LogicCompareNum""]","The instructor's tags are a good start, but th...",3,5,2,1.94,"```json\n{\n ""problem_id"": 5,\n ""added_k..."
3,12,"[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...","[""NestedIf"", ""LogicAndNotOr"", ""LogicCompareNum...",[],The instructor's tags are sufficient. The fail...,5,5,0,1.47,"```json\n{\n ""problem_id"": 12,\n ""added_..."
4,13,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""NestedIf"", ""Math+-*/"", ""LogicAndNotOr"", ""Log...",[],The existing KCs seem sufficient. The failing ...,6,6,0,1.26,"```json\n{\n ""problem_id"": 13,\n ""added_..."
5,17,"[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...","[""If/Else"", ""NestedIf"", ""LogicAndNotOr"", ""Logi...",[],The instructor's tags seem sufficient. The fai...,4,4,0,1.45,"```json\n{\n ""problem_id"": 17,\n ""added_..."
6,20,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""NestedIf"", ""Math+-*/"", ""LogicAndNotOr"", ""Log...",[],The existing tags adequately cover the knowled...,5,5,0,1.38,"```json\n{\n ""problem_id"": 20,\n ""added_..."
7,21,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""NestedIf"", ""Math+-*/"", ""LogicAndNotOr"", ""Log...","[""If/Else""]","The instructor's tags are mostly sufficient, b...",5,5,1,2.49,"```json\n{\n ""problem_id"": 21,\n ""added_..."
8,22,"[""If/Else"", ""NestedIf"", ""Math+-*/"", ""LogicAndN...","[""NestedIf"", ""Math+-*/"", ""DefFunction"", ""Logic...",[],The existing KCs seem sufficient. The failing ...,6,6,0,1.18,"```json\n{\n ""problem_id"": 22,\n ""added_..."
9,24,"[""If/Else"", ""Math+-*/"", ""LogicAndNotOr"", ""Logi...","[""Math+-*/"", ""Math%"", ""LogicAndNotOr"", ""LogicC...","[""Math%""]",The instructor's current tags are mostly suffi...,4,5,1,1.28,"```json\n{\n ""problem_id"": 24,\n ""added_..."


### ✅ [Exp No. 11] Human-LLM Agreement Analysis

**Purpose Of Exp:**
Calculate Cohen's Kappa and other agreement metrics between human annotators and our LLM-based assessment to quantify reliability and human-expert level performance.

**Conclusion:**
LLM assessment reached "Moderate" to "Substantial" agreement with human raters on core Java skills like loops and conditionals. Agreement was slightly lower on more abstract logic concepts.


In [8]:
# Results for Exp 11: Human-LLM Agreement Analysis
display_file('human_llm_agreement/agreement_metrics_table.csv')
display_file('human_llm_agreement/agreement_metrics_by_cluster.csv')
display_file('human_validation/human_validation_results.json')
display_file('human_validation/exp11_enriched_v2_vs_human_metrics.json')


FILE: human_llm_agreement/agreement_metrics_table.csv
--------------------------------------------------------------------------------


,Comparison,Cohen_kappa,Gwet_AC1,Problem_F1,Jaccard
0,H-A vs H-B (Ceiling),0.574,0.953,0.869,0.834
1,AvgHuman vs Baseline,0.344,0.906,0.741,0.708
2,AvgHuman vs Enriched,0.373,0.934,0.784,0.754
3,AvgHuman vs Baseline_V2,0.426,0.932,0.811,0.781
4,AvgHuman vs Enriched_V2,0.406,0.927,0.798,0.764
5,AvgHuman vs V3,0.509,0.952,0.842,0.811



FILE: human_llm_agreement/agreement_metrics_by_cluster.csv
--------------------------------------------------------------------------------


,Cluster,Student,N_Problems,Comparison,Cohen_kappa,Gwet_AC1,Problem_F1,Jaccard,Note
0,Struggling,10155,46,Human_A vs Human_B,0.673152,0.957720,0.890269,0.850880,NaN
1,Struggling,10155,46,AvgHuman vs Exp10a_Baseline,0.357159,0.877175,0.656513,0.613147,NaN
2,Struggling,10155,46,AvgHuman vs Exp10b_Enriched,0.465361,0.932690,0.785438,0.741175,NaN
3,Struggling,10155,46,AvgHuman vs Exp11_Baseline_V2,0.475141,0.923165,0.794919,0.756030,NaN
4,Struggling,10155,46,AvgHuman vs Exp11_Enriched_V2,0.419301,0.906835,0.763276,0.721196,NaN
5,Average,14476,50,Human_A vs Human_B,0.523782,0.903910,0.778476,0.711143,NaN
6,Average,14476,50,AvgHuman vs Exp10a_Baseline,0.319600,0.844235,0.642989,0.588790,NaN
7,Average,14476,50,AvgHuman vs Exp10b_Enriched,0.304268,0.873830,0.651056,0.603992,NaN
8,Average,14476,50,AvgHuman vs Exp11_Baseline_V2,0.401118,0.882970,0.722766,0.674401,NaN
9,Average,14476,50,AvgHuman vs Exp11_Enriched_V2,0.388185,0.878695,0.711229,0.655595,NaN



FILE: human_validation/human_validation_results.json
--------------------------------------------------------------------------------
student_id                    : 10155
n_problems                    : 46
human_a                       : Pranay Ghuge
human_b                       : Arundhati Das
llm                           : LLM_Gemini_Enriched
llm_vs_human_ratio            : 0.6913155845002777

[pairwise]


,human_a_vs_human_b,human_a_vs_llm,human_b_vs_llm
kappa,0.673152,0.489204,0.441517
f1,0.693069,0.520000,0.474227
precision,0.714286,0.541667,0.479167
recall,0.673077,0.500000,0.469388



FILE: human_validation/exp11_enriched_v2_vs_human_metrics.json
--------------------------------------------------------------------------------
num_problems                  : 146

[human_vs_human]
{
    "name": "Human A vs Human B",
    "kappa": 0.42098942446437365,
    "f1": 0.4593837535014006,
    "precision": 0.39805825242718446,
    "recall": 0.543046357615894
}

[average_vs_enriched]
{
    "kappa": 0.35366933667410555,
    "f1": 0.3964595242485709
}

[average_vs_baseline]
{
    "kappa": 0.37141975536602206,
    "f1": 0.41200604431324994
}

[details]


,human_a_vs_baseline,human_b_vs_baseline,human_a_vs_enriched,human_b_vs_enriched
name,Human A vs LLM Baseline,Human B vs LLM Baseline,Human A vs LLM Enriched,Human B vs LLM Enriched
kappa,0.470291,0.272549,0.449439,0.2579
f1,0.501608,0.322404,0.482759,0.31016
precision,0.4875,0.36875,0.458333,0.345238
recall,0.516556,0.286408,0.509934,0.281553


### 🧪 [Exp No. 12] V3 Prompt Evaluation Results (Filtered Gap Cases)

**Purpose Of Exp:**
Compare Human A, Human B, and LLM V3 annotations on the 10-student validation set. Problems where all three raters marked no gaps are skipped, so the evaluation focuses on cases where at least one rater identified a KC gap.

**Conclusion:**
LLM V3 reaches 84.4% of the human F1 ceiling and 80.8% of the human Kappa ceiling on the filtered gap-case set.


In [9]:
# Results for Exp 12: V3 Prompt Evaluation Results (Filtered Gap Cases)

overall_path = find_results_path('human_validation/v3_prompt_eval_results/overall_metrics.csv')
ceiling_path = find_results_path('human_validation/v3_prompt_eval_results/ceiling_metrics.csv')

overall_df = pd.read_csv(overall_path)
comparison_table = overall_df[['Pair', 'N Problems', 'F1 Score', 'Jaccard Value', 'Kappa', 'Gwet AC1']].rename(columns={
    'Pair': 'Comparison',
    'N Problems': 'Problems',
    'F1 Score': 'F1',
    'Jaccard Value': 'Problem Jaccard',
})

print('=' * 80)
print('V3 Comparison Table')
print('-' * 80)
display(comparison_table.style.format({
    'F1': '{:.3f}',
    'Problem Jaccard': '{:.3f}',
    'Kappa': '{:.3f}',
    'Gwet AC1': '{:.3f}',
}))

ceiling_df = pd.read_csv(ceiling_path)
summary_table = ceiling_df.rename(columns={
    'LLM Avg vs Humans': 'LLM Avg',
    'Gap to Ceiling': 'Gap',
    'Percent of Ceiling': '% of Ceiling',
})

print('=' * 80)
print('V3 Summarized Results')
print('-' * 80)
display(summary_table.style.format({
    'Human Ceiling': '{:.3f}',
    'LLM Avg': '{:.3f}',
    'Gap': '{:.3f}',
    '% of Ceiling': '{:.1%}',
}))
print('=' * 80)


V3 Comparison Table
--------------------------------------------------------------------------------


,Comparison,Problems,F1,Problem Jaccard,Kappa,Gwet AC1
0,Human A vs Human B,135,0.687,0.576,0.633,0.879
1,Human A vs LLM V3,135,0.600,0.480,0.534,0.850
2,Human B vs LLM V3,135,0.558,0.435,0.489,0.844
3,LLM V3 average vs humans,135,0.579,0.458,0.512,0.847


V3 Summarized Results
--------------------------------------------------------------------------------


,Metric,Human Ceiling,LLM Avg,Gap,% of Ceiling
0,F1 Score,0.687,0.579,0.107,84.4%
1,Jaccard Value,0.576,0.458,0.119,79.4%
2,Kappa,0.633,0.512,0.122,80.8%
3,Gwet AC1,0.879,0.847,0.032,96.3%
